In [1]:
!pip install -q groq

In [15]:
from groq import Groq
from google.colab import userdata
import json

api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

product_name = input("Enter the product name: ")

prompt = f"""
Generate details for the product "{product_name}".

Return ONLY valid JSON.

The JSON must contain exactly these fields:
- name
- description
- price
- category

Do not include markdown.
Do not include explanations.
Do not include code fences.
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3,
    max_tokens=200
)

response_text = response.choices[0].message.content

print("Raw Response:\n")
print(response_text)

product_data = json.loads(response_text)


print(type(product_data))



Enter the product name: IPhone 16
Raw Response:

{
  "name": "iPhone 16",
  "description": "A high-performance smartphone with advanced camera capabilities and long-lasting battery life",
  "price": 999.99,
  "category": "Electronics"
}
<class 'dict'>


In [16]:
required_fields = [
    "name",
    "description",
    "price",
    "category"
]

try:
    product_data = json.loads(response_text)

    for field in required_fields:
        if field not in product_data:
            raise ValueError(f"Missing field: {field}")

    print("JSON is valid.\n")
    print(product_data)

except json.JSONDecodeError:
    print("Error: Invalid JSON format.")

except ValueError as e:
    print(e)

except Exception as e:
    print("Unexpected Error:", e)

JSON is valid.

{'name': 'iPhone 16', 'description': 'A high-performance smartphone with advanced camera capabilities and long-lasting battery life', 'price': 999.99, 'category': 'Electronics'}


In [17]:
improved_prompt = f"""
Generate information about the product "{product_name}".

Return ONLY a valid JSON object.

Rules:
1. Output must be valid JSON.
2. Do not use markdown.
3. Do not use code fences.
4. Do not write any explanation.
5. Include all of these fields:
   - name
   - description
   - price
   - category
6. Price must be numeric.
7. Category must be a string.
"""

response2 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": improved_prompt
        }
    ],
    temperature=0.1,
    max_tokens=200
)

improved_response = response2.choices[0].message.content
print(improved_response)



{
  "name": "iPhone 16",
  "description": "The latest iPhone model with advanced features and improved performance",
  "price": 999,
  "category": "Smartphones"
}


In [9]:
try:
    improved_data = json.loads(improved_response)

    for field in required_fields:
        if field not in improved_data:
            raise ValueError(f"Missing field: {field}")

    print("Improved JSON is valid.\n")

    print(improved_data)

except Exception as e:
    print(e)

Improved JSON is valid.

{'name': 'iPhone 16', 'description': 'The latest flagship smartphone from Apple, featuring advanced camera capabilities, improved performance, and enhanced security features', 'price': 999, 'category': 'Electronics'}


# JSON Parsing Challenges and Solutions

## Challenges Faced

1. The LLM may return text along with the JSON response.
2. Responses may include Markdown code blocks (```json).
3. Some required fields may be missing.
4. The generated JSON may have syntax errors such as missing commas or quotation marks.

## How These Challenges Were Resolved

- The prompt was refined to explicitly request only valid JSON.
- Markdown formatting and explanations were prohibited.
- Python's `json.loads()` function was used to parse the response.
- `try-except` blocks were implemented to catch JSON parsing errors.
- Validation was performed to ensure that all required fields (`name`, `description`, `price`, and `category`) were present in the response.
- A second, more restrictive prompt was used to regenerate cleaner and more reliable JSON output.